In [1]:
# -*- coding: utf-8 -*-
"""
BigAlpha 2026 端到端量价预测 —— 推理脚本 v5

结构对齐原版 Transformer_modelsave_predict.py。
提交时转为 .ipynb，平台调用 main(datasources, start_date, end_date)。
"""
import os
import numpy as np
import pandas as pd
import dai
import torch
import structlog

from transformer_train_v2 import (
    FREQ_CONFIGS, FREQ_LABELS, FREQ_TABLES, MODEL_CFG,
    MultiFreqStockTransformer, SimpleStockTransformer,
    load_model_json, count_parameters,
)

logger = structlog.get_logger()
MODEL_PATH = os.path.join(os.getcwd(), "transformer_model_v2.json")
INSTRUMENT_TABLE = "bigalpha_2026_instruments"


def query_instruments(sd, ed):
    df = dai.query(
        f"SELECT DISTINCT instrument FROM {INSTRUMENT_TABLE}",
        filters={"date": [sd, ed]}
    ).df()
    return sorted(df["instrument"].tolist())


def query_freq(table, sd, ed, instruments, columns):
    buf = (pd.to_datetime(sd) - pd.Timedelta(days=120)).strftime("%Y-%m-%d")
    sql = f"SELECT date, instrument, {', '.join(columns)} FROM {table} ORDER BY instrument, date"
    df = dai.query(sql, filters={"date": [buf, ed], "instrument": instruments}).df()
    for c in columns:
        if "volume" in c.lower() or c == "amount":
            if c in df.columns: df[c] = np.log1p(df[c].clip(lower=0))
    return df


def build_seqs(df, target_dates, seq_len, columns):
    result = {}
    doi = set(pd.to_datetime(target_dates))
    for ins, sub in df.groupby("instrument", sort=False):
        sub = sub.sort_values("date").reset_index(drop=True)
        if len(sub) <= seq_len: continue
        feats = sub[columns].to_numpy(np.float32)
        day_ts = sub["date"].dt.normalize().to_numpy()
        cp = np.flatnonzero(np.append(day_ts[1:] != day_ts[:-1], True))
        dates_arr = day_ts[cp]
        ir = {}
        for k, p in enumerate(cp):
            d = pd.Timestamp(dates_arr[k])
            if p + 1 >= seq_len and d in doi:
                ir[d] = feats[p-seq_len+1:p+1].copy()
        if ir: result[ins] = ir
    return result


def main(datasources, start_date, end_date):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---- 1. 加载模型 ----
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"未找到模型权重: {MODEL_PATH}")

    ckpt = load_model_json(MODEL_PATH, map_location=device)
    use_multi_freq = ckpt.get("use_multi_freq", True)

    import inspect
    if use_multi_freq:
        valid_keys = set(inspect.signature(MultiFreqStockTransformer.__init__).parameters.keys()) - {"self"}
    else:
        valid_keys = set(inspect.signature(SimpleStockTransformer.__init__).parameters.keys()) - {"self"}

    cfg = {k: v for k, v in MODEL_CFG.items() if k in valid_keys}
    saved_cfg = ckpt.get("model_cfg", {}) or {}
    cfg.update({k: v for k, v in saved_cfg.items() if k in valid_keys})

    if use_multi_freq:
        model = MultiFreqStockTransformer(**cfg).to(device)
    else:
        model = SimpleStockTransformer(**cfg).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()

    stats = {k: (np.array(v[0], np.float32), np.array(v[1], np.float32))
             for k, v in ckpt["stats"].items()}

    n_params = sum(p.numel() for p in model.parameters())
    logger.info("模型已加载", params=f"{n_params:,}")

    # ---- 2. 可用频率 ----
    active = []
    for fl in FREQ_LABELS:
        if datasources.get(f"bar{fl}"): active.append(fl)
    if not active: active = [FREQ_LABELS[0]]

    # ---- 3. 准备数据 ----
    ds, de = str(start_date)[:10], str(end_date)[:10]
    instruments = [i for i in query_instruments(ds, de) if pd.notna(i)]
    logger.info("成分股", n=len(instruments))

    first_table = datasources.get("bar1m", FREQ_TABLES["1m"])
    tdf = dai.query(
        f"SELECT DISTINCT date FROM {first_table}",
        filters={"date": [ds, de], "instrument": instruments[:5]}
    ).df()
    trading_days = (sorted(tdf["date"].dt.normalize().unique()) if not tdf.empty
                    else pd.bdate_range(start=ds, end=de))

    # ---- 4. 查询各频率数据 ----
    all_seqs = {}
    for fl in active:
        cfg = FREQ_CONFIGS[fl]
        table = datasources.get(f"bar{fl}", FREQ_TABLES[fl])
        cols = cfg["all_cols"]
        logger.info(f"查询 {fl} ({len(cols)}字段)...")
        df = query_freq(table, start_date, end_date, instruments, cols)
        all_seqs[fl] = build_seqs(df, trading_days, cfg["seq_len"], cols)

    # ---- 5. 逐日推理 ----
    logger.info("逐日推理", n_days=len(trading_days))
    all_results = []

    for date in trading_days:
        d = pd.Timestamp(date)
        day_ins, day_xs = [], {fl: [] for fl in active}

        for ins in instruments:
            ok = all(ins in all_seqs[fl] and d in all_seqs[fl][ins] for fl in active)
            if not ok: continue
            for fl in active: day_xs[fl].append(all_seqs[fl][ins][d])
            day_ins.append(ins)

        if len(day_ins) < 2: continue

        xs = []
        for fl in active:
            m, s = stats[fl]
            xs.append(torch.from_numpy(np.stack([(seq-m)/s for seq in day_xs[fl]]).astype(np.float32)))

        with torch.no_grad():
            if use_multi_freq:
                h = model.encode_freqs([x.to(device) for x in xs])
                h = h.unsqueeze(0)
                h = model.cross_stock(h)
                h = h.squeeze(0)
                scores = model.score_head(h).squeeze(-1).cpu().numpy()
            else:
                h = model.encode(xs[0].to(device))
                h = h.unsqueeze(0)
                h = model.cross_stock(h)
                h = h.squeeze(0)
                scores = model.score_head(h).squeeze(-1).cpu().numpy()

        for ins, sc in zip(day_ins, scores):
            all_results.append({"date": d, "instrument": ins, "score": float(sc)})

    # ---- 6. 对齐 + 输出 ----
    stk = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [ds, de]}
    ).df()

    result = (pd.DataFrame(all_results)
              .merge(stk, on=["date", "instrument"], how="inner")
              .replace([np.inf, -np.inf], np.nan).dropna(subset=["score"])
              .drop_duplicates(["date", "instrument"])
              [["date", "instrument", "score"]].reset_index(drop=True))

    logger.info("完成", rows=len(result), days=result["date"].nunique())
    return result


if __name__ == "__main__":
    from bigmodule import M
    datasources = {"bar1m":FREQ_TABLES["1m"],"bar5m":FREQ_TABLES["5m"],
                   "bar15m":FREQ_TABLES["15m"],"bar30m":FREQ_TABLES["30m"]}
    if not os.path.exists(MODEL_PATH):
        from transformer_train_v2 import train_and_save
        train_and_save(datasources)
    sd, ed = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
    sc = main(datasources, sd, ed)
    print(sc.head(10))
    M.bigalpha_eval._latest(factor_data=sc, show=True)


[2026-07-06 15:03:45] [info     ] 设备                             device=cuda
[2026-07-06 15:03:45] [info     ] 成分股                            train=500 val=332
[2026-07-06 15:03:45] [info     ] 1m: 28字段, seq_len=240
[2026-07-06 15:05:33] [info     ] 5m: 20字段, seq_len=200
[2026-07-06 15:05:59] [info     ] 15m: 12字段, seq_len=160
[2026-07-06 15:06:09] [info     ] 30m: 7字段, seq_len=120
[2026-07-06 15:06:20] [info     ] 对齐样本                           common=355250 counts=[355460, 355400, 355325, 355250]
[2026-07-06 15:06:24] [info     ] BARRA 暴露数据不可用，使用原始收益标签
[2026-07-06 15:06:24] [info     ] 数据划分                           train_days=665 val_days=60
[2026-07-06 15:07:23] [info     ] 参数量                            n_params=865925
[2026-07-06 15:08:48] [info     ] Epoch 1/30                     loss=0.358215 lr=0.00015 val_ic=0.066312
[2026-07-06 15:08:49] [info     ] 保存最佳                           ic=0.066312
[2026-07-06 15:10:14] [info     ] Epoch 2/30                     loss=0.347985 lr=0